# Sprint 14 — Sure Tomorrow (entrega)

**Sure Tomorrow** es una compañía de seguros que quiere:
1. encontrar clientes similares a un cliente dado (marketing / look-alikes);
2. predecir si un cliente recibirá un beneficio de seguro (clasificación binaria);
3. predecir el número de beneficios con una regresión lineal propia;
4. proteger los datos personales ofuscando las características con una matriz invertible, sin alterar las predicciones lineales.

Este cuaderno es **autocontenido** (sin `sys.path` ni imports de `src/`). Semilla fija **42** en todos los sorteos, la matriz \(P\) y las particiones train/test.

| Tarea | Contenido |
|-------|-----------|
| **T1** | Similitud: vecinos \(k=5\) con Euclidiana / Manhattan × crudo / MaxAbs |
| **T2** | Clasificación kNN \(k=1\ldots10\) vs cuatro baselines dummy |
| **T3** | OLS por ecuación normal; RMSE en hold-out 30%; residuos |
| **T4** | Ofuscación \(X\mapsto XP\); recuperación con \(P^{-1}\); RMSE y \(R^2\) |

### Advertencias explícitas (antes de las tareas)
1. La **regresión lineal** sobre un conteo inflado en cero y escalonado por edad produce predicciones negativas o fraccionarias imposibles; los diagnósticos de residuales lo hacen visible.
2. La **distancia euclidiana sin escalar** está dominada por el salario y no es usable para marketing.
3. La **ofuscación matricial** solo protege contra inspección casual; quien conozca \(P\) recupera los datos originales (salvo ruido de punto flotante).
4. **No** se hacen afirmaciones de selección de modelo, producción ni cumplimiento normativo.


## Imports y procedimientos (en el cuaderno)

Ruta de datos: se prefiere `/datasets/insurance_us.csv`; si no existe, se usa el CSV del repositorio `data/insurance_us.csv`.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)

# Prefer Tripleten path; fall back to repo data/ for local smoke checks.
_CANDIDATES = [
    Path("/datasets/insurance_us.csv"),
    Path.cwd().resolve() / "data" / "insurance_us.csv",
    Path.cwd().resolve().parent / "data" / "insurance_us.csv",
]
DATA_PATH = next((p for p in _CANDIDATES if p.exists()), _CANDIDATES[0])
print("numpy", np.__version__)
print("data path:", DATA_PATH, "| exists:", DATA_PATH.exists())

COLUMN_NAMES = ["gender", "age", "salary", "family_members"]
_RENAME = {
    "Gender": "gender",
    "Age": "age",
    "Salary": "salary",
    "Family members": "family_members",
    "Insurance benefits": "insurance_benefits",
}
_QUANTILES = [0.0, 0.25, 0.5, 0.75, 1.0]


@dataclass(frozen=True)
class NeighborResult:
    index: int
    features: np.ndarray
    distance: float
    shares: np.ndarray


@dataclass(frozen=True)
class OLSResult:
    weights: np.ndarray
    predictions: np.ndarray
    rmse: float


@dataclass(frozen=True)
class ObfuscationResult:
    P: np.ndarray
    X_obf: np.ndarray
    X_recovered: np.ndarray
    pred_diff: float
    rmse_diff: float
    rmse_orig: float
    rmse_obf: float
    r2_orig: float
    r2_obf: float


def load_arrays(
    path: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], Dict[str, object]]:
    """Carga el CSV; age como entero en origen, float64 en la matriz de rasgos."""
    df = pd.read_csv(path)
    df = df.rename(columns=_RENAME)
    df = df.rename(columns=lambda c: str(c).lower().replace(" ", "_"))
    if "age" in df.columns and len(df) > 0:
        df["age"] = df["age"].astype(int)

    n = int(len(df))
    if n == 0:
        X = np.empty((0, 4), dtype=np.float64)
        y = np.empty((0,), dtype=int)
        y_bin = np.empty((0,), dtype=int)
        stats = {
            "age_quantiles": np.array([], dtype=np.float64),
            "salary_quantiles": np.array([], dtype=np.float64),
            "n": 0,
            "positive_rate": 0.0,
        }
        return X, y, y_bin, list(COLUMN_NAMES), stats

    X = df[COLUMN_NAMES].to_numpy(dtype=np.float64)
    y = df["insurance_benefits"].to_numpy(dtype=int)
    y_bin = (y > 0).astype(int)
    stats = {
        "age_quantiles": np.quantile(X[:, 1], _QUANTILES),
        "salary_quantiles": np.quantile(X[:, 2], _QUANTILES),
        "n": n,
        "positive_rate": float(y_bin.mean()),
    }
    return X, y, y_bin, list(COLUMN_NAMES), stats


def _maxabs_scale(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    scale = np.max(np.abs(X), axis=0)
    scale = np.where(scale == 0.0, 1.0, scale)
    return X / scale, scale


def _contribution_shares(delta: np.ndarray, metric: str) -> np.ndarray:
    pieces = delta ** 2 if metric == "euclidean" else np.abs(delta)
    total = pieces.sum()
    if total == 0.0:
        return np.zeros_like(pieces, dtype=np.float64)
    return pieces / total


def _k_neighbors(X, X_space, query_space, k, metric):
    if metric == "euclidean":
        dists = np.sqrt(np.sum((X_space - query_space) ** 2, axis=1))
    else:
        dists = np.sum(np.abs(X_space - query_space), axis=1)
    n = X_space.shape[0]
    if n == 0:
        return []
    kk = min(k, n)
    part = np.argpartition(dists, kk - 1)[:kk]
    order = part[np.argsort(dists[part], kind="mergesort")]
    results = []
    for idx in order:
        i = int(idx)
        delta = X_space[i] - query_space
        results.append(
            NeighborResult(
                index=i,
                features=np.asarray(X[i], dtype=np.float64).copy(),
                distance=float(dists[i]),
                shares=np.asarray(_contribution_shares(delta, metric), dtype=np.float64),
            )
        )
    return results


def distances(X, queries, k: int = 5):
    """Cuatro escenarios: euclidean/manhattan × raw/scaled. Consultas las pasa el llamador."""
    X = np.asarray(X, dtype=np.float64)
    X_scaled, scale = _maxabs_scale(X)
    keys = ("euclidean_raw", "euclidean_scaled", "manhattan_raw", "manhattan_scaled")
    out = {key: [] for key in keys}
    for q in queries:
        q = np.asarray(q, dtype=np.float64).reshape(4,)
        q_scaled = q / scale
        out["euclidean_raw"].append(_k_neighbors(X, X, q, k, "euclidean"))
        out["euclidean_scaled"].append(_k_neighbors(X, X_scaled, q_scaled, k, "euclidean"))
        out["manhattan_raw"].append(_k_neighbors(X, X, q, k, "manhattan"))
        out["manhattan_scaled"].append(_k_neighbors(X, X_scaled, q_scaled, k, "manhattan"))
    return out


def _design_matrix(X: np.ndarray) -> np.ndarray:
    return np.hstack([np.ones((X.shape[0], 1), dtype=np.float64), X])


def _maxabs_features(X: np.ndarray) -> np.ndarray:
    scale = np.max(np.abs(X), axis=0)
    scale = np.where(scale == 0.0, 1.0, scale)
    return X / scale


def linreg_ols(X, y, scale: bool = False) -> OLSResult:
    """OLS por ecuación normal (lstsq). Predicciones reales sin truncar."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    feats = _maxabs_features(X) if scale else X
    A = _design_matrix(feats)
    weights, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
    predictions = A @ weights
    rmse = float(np.sqrt(np.mean((y - predictions) ** 2))) if len(y) else 0.0
    return OLSResult(
        weights=np.asarray(weights, dtype=np.float64),
        predictions=np.asarray(predictions, dtype=np.float64),
        rmse=rmse,
    )


def _r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    if ss_tot == 0.0:
        return 0.0
    return 1.0 - ss_res / ss_tot


def _f1_and_cm(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    n = len(y_true)
    if n == 0:
        return {"f1": 0.0, "tn": 0.0, "fp": 0.0, "fn": 0.0, "tp": 0.0}
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {"f1": float(f1), "tn": tn / n, "fp": fp / n, "fn": fn / n, "tp": tp / n}


def dummies(y_bin, seed: int = 42):
    """Cuatro baselines Bernoulli: p=0, tasa empírica, 0.5, 1."""
    y_bin = np.asarray(y_bin, dtype=int)
    n = len(y_bin)
    rate = float(y_bin.mean()) if n else 0.0
    probs = {"p_0": 0.0, "p_rate": rate, "p_0.5": 0.5, "p_1": 1.0}
    rng = np.random.RandomState(seed)
    out = {}
    for name, p in probs.items():
        if n == 0:
            draws = np.empty((0,), dtype=int)
        elif p <= 0.0:
            draws = np.zeros(n, dtype=int)
        elif p >= 1.0:
            draws = np.ones(n, dtype=int)
        else:
            draws = rng.binomial(1, p, size=n)
        out[name] = _f1_and_cm(y_bin, draws)
    return out


def knn_f1_sweep(X_train, y_train, X_test, y_test, ks=range(1, 11)):
    """F1 de kNN euclidiano puro NumPy para cada k."""
    X_train = np.asarray(X_train, dtype=np.float64)
    X_test = np.asarray(X_test, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=int)
    y_test = np.asarray(y_test, dtype=int)
    train_sq = np.sum(X_train ** 2, axis=1)
    test_sq = np.sum(X_test ** 2, axis=1)
    cross = X_test @ X_train.T
    dist2 = test_sq[:, None] + train_sq[None, :] - 2.0 * cross
    order = np.argsort(dist2, axis=1, kind="mergesort")
    f1s = []
    for k in ks:
        votes = y_train[order[:, :k]]
        pred = (votes.mean(axis=1) >= 0.5).astype(int)
        f1s.append(_f1_and_cm(y_test, pred)["f1"])
    return f1s


def _draw_invertible(seed: int = SEED, size: int = 4) -> np.ndarray:
    rng = np.random.RandomState(seed)
    while True:
        P = rng.normal(size=(size, size))
        if abs(np.linalg.det(P)) > 1e-8:
            return np.asarray(P, dtype=np.float64)


def obfuscate(X, y) -> ObfuscationResult:
    """Aplica P invertible (seed 42) al bloque de rasgos; demuestra identidad OLS y recuperación."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)
    P = _draw_invertible(SEED)
    X_obf = X @ P
    ols_orig = linreg_ols(X, y, scale=False)
    ols_obf = linreg_ols(X_obf, y, scale=False)
    X_recovered = X_obf @ np.linalg.inv(P)
    return ObfuscationResult(
        P=P,
        X_obf=np.asarray(X_obf, dtype=np.float64),
        X_recovered=np.asarray(X_recovered, dtype=np.float64),
        pred_diff=float(np.max(np.abs(ols_orig.predictions - ols_obf.predictions))),
        rmse_diff=float(abs(ols_orig.rmse - ols_obf.rmse)),
        rmse_orig=float(ols_orig.rmse),
        rmse_obf=float(ols_obf.rmse),
        r2_orig=_r2(y, ols_orig.predictions),
        r2_obf=_r2(y, ols_obf.predictions),
    )


def f1_vs_k(f1_raw, f1_scaled):
    k_raw = list(range(1, len(f1_raw) + 1))
    k_scaled = list(range(1, len(f1_scaled) + 1))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(k_raw, f1_raw, marker="o", label="crudo")
    ax.plot(k_scaled, f1_scaled, marker="s", label="MaxAbs")
    ax.set_xlabel("k")
    ax.set_ylabel("F1")
    ax.set_title("F1 frente a k (clasificador kNN)")
    ax.set_xticks(sorted(set(k_raw) | set(k_scaled)))
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig


def residuals_vs_fitted(y, predictions):
    y = np.asarray(y, dtype=np.float64)
    predictions = np.asarray(predictions, dtype=np.float64)
    resid = y - predictions
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(predictions, resid, alpha=0.35, s=12)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xlabel("Ajuste (predicción OLS cruda)")
    ax.set_ylabel("Residual")
    ax.set_title("Residuales vs ajuste (OLS — sin truncar)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig


def age_binned_residual_means(age, residuals, bins=None):
    age = np.asarray(age, dtype=np.float64)
    residuals = np.asarray(residuals, dtype=np.float64)
    if bins is None:
        bins = np.arange(age.min(), age.max() + 5, 5)
    digitize = np.digitize(age, bins)
    centers, means = [], []
    for b in range(1, len(bins)):
        mask = digitize == b
        if not np.any(mask):
            continue
        centers.append(0.5 * (bins[b - 1] + bins[b]))
        means.append(float(residuals[mask].mean()))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(centers, means, width=3.5, align="center")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xlabel("Edad (centro del bin)")
    ax.set_ylabel("Residual medio")
    ax.set_title("Medias de residuales por bin de edad")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    return fig


def pairplot(X, y=None, column_names=None):
    names = list(column_names) if column_names is not None else list(COLUMN_NAMES)
    data = {names[i]: X[:, i] for i in range(X.shape[1])}
    if y is not None:
        data["insurance_benefits"] = np.asarray(y)
    g = sns.pairplot(pd.DataFrame(data), kind="hist")
    g.fig.set_size_inches(12, 12)
    return g


def correlation_heatmap(X, y=None, column_names=None):
    names = list(column_names) if column_names is not None else list(COLUMN_NAMES)
    data = {names[i]: X[:, i] for i in range(X.shape[1])}
    if y is not None:
        data["insurance_benefits"] = np.asarray(y)
    corr = pd.DataFrame(data).corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(corr, annot=True, cmap="coolwarm", ax=ax)
    ax.set_title("Mapa de calor de correlaciones")
    fig.tight_layout()
    return fig


def as_plain(names, values, ndigits=None):
    out = {}
    for k, v in zip(names, values):
        x = float(v)
        out[k] = round(x, ndigits) if ndigits is not None else x
    return out

print("Procedimientos listos.")


## Revisión de datos (P3)

Comprobaciones visibles: valores faltantes, duplicados, tipos (`dtypes`), `describe` por columna. Sin descartes silenciosos: los duplicados se reportan; no se eliminan aquí para no alterar conteos de las tareas.


In [ ]:
# Carga con pandas solo para la revisión visible; luego NumPy puro vía load_arrays.
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw.rename(columns=_RENAME)
df_raw = df_raw.rename(columns=lambda c: str(c).lower().replace(" ", "_"))

print("Forma:", df_raw.shape)
print("\nTipos (dtypes):")
print(df_raw.dtypes)
print("\nValores faltantes por columna:")
print(df_raw.isna().sum())
print("\nDuplicados exactos (filas):", int(df_raw.duplicated().sum()))
print("\nDescribe por columna:")
print(df_raw.describe().T.round(2))
print("\nConteo de insurance_benefits:")
print(df_raw["insurance_benefits"].value_counts().sort_index())

X, y, y_bin, cols, stats = load_arrays(str(DATA_PATH))
print("\nTras load_arrays → NumPy:")
print("columnas:", cols)
print("X", X.shape, X.dtype, "| y", y.shape, "| y_bin", y_bin.shape)
print("n =", stats["n"], "| tasa positiva =", round(stats["positive_rate"], 4))
print("cuantiles edad:", np.round(stats["age_quantiles"], 2))
print("cuantiles salario:", np.round(stats["salary_quantiles"], 2))


### Conclusión de la revisión

Los datos son **usables** para las cuatro tareas: no hay valores faltantes; los tipos son coherentes (género binario, edad entera, salario continuo, familiares enteros, beneficios enteros \(0\ldots5\)).

**Matices / posibles extremos a tener en cuenta:**
- el salario cubre un rango amplio (aprox. 5 300–79 000), lo que domina distancias sin escalar;
- los beneficios llegan hasta 5, pero la masa está en 0 (inflación de ceros);
- hay filas duplicadas exactas; se dejan visibles y no se eliminan en silencio (las tareas siguientes usan la matriz tal cual la carga `load_arrays`).

Tras esta revisión, el resto del cuaderno trabaja en NumPy puro.


## Exploración gráfica

El pair plot muestra beneficios concentrados en 0 y un escalón por edad (~desde los 40). El mapa de calor confirma correlación edad–beneficios ≈ 0.65; el resto cerca de cero.


In [ ]:
g = pairplot(X, y, cols)
plt.show()

fig_corr = correlation_heatmap(X, y, cols)
plt.show()


## T1 — Similitud (P4)

Claves: `euclidean_raw`, `euclidean_scaled`, `manhattan_raw`, `manhattan_scaled`. \(k=5\).

Aportes (shares): Manhattan \(|\Delta_i|/\sum|\Delta_j|\); Euclidiana \((\Delta_i)^2/\sum(\Delta_j)^2\).

Las consultas son **filas reales** del conjunto (no sintéticas fuera de rango).


In [ ]:
# Tres clientes reales como consultas (índices fijos para reproducibilidad).
query_indices = [0, 100, 250]
# Si algún índice cae fuera (n pequeño), recorta; el CSV oficial tiene 5000.
query_indices = [i for i in query_indices if i < X.shape[0]]
queries = [X[i].copy() for i in query_indices]

print("Consultas = filas reales del dataset:")
for qi, idx in enumerate(query_indices):
    print(f"  query{qi+1} ← fila {idx}:", as_plain(COLUMN_NAMES, queries[qi]))

nn = distances(X, queries, k=5)
assert set(nn.keys()) == {
    "euclidean_raw", "euclidean_scaled", "manhattan_raw", "manhattan_scaled"
}


def show_neighbors(scenario, query_idx):
    print("=" * 70)
    print(f"{scenario} | consulta {query_idx + 1} (fila {query_indices[query_idx]})")
    for rank, nb in enumerate(nn[scenario][query_idx], 1):
        print(f"  #{rank} idx={nb.index} dist={nb.distance:.4f}")
        print(f"      rasgos ={as_plain(COLUMN_NAMES, nb.features)}")
        print(
            f"      shares ={as_plain(COLUMN_NAMES, nb.shares, ndigits=4)}"
            f"  (suma={float(nb.shares.sum()):.4f})"
        )


for scenario in nn:
    for qi in range(len(queries)):
        show_neighbors(scenario, qi)

# Comparación de índices de vecinos: Manhattan vs Euclidiana, crudo y escalado.
print("\n" + "=" * 70)
print("Comparación de índices de vecinos (sin el propio índice si aparece como dist=0)")
for qi in range(len(queries)):
    er = [nb.index for nb in nn["euclidean_raw"][qi]]
    mr = [nb.index for nb in nn["manhattan_raw"][qi]]
    es = [nb.index for nb in nn["euclidean_scaled"][qi]]
    ms = [nb.index for nb in nn["manhattan_scaled"][qi]]
    print(f"\nConsulta {qi+1} (fila {query_indices[qi]}):")
    print(f"  euclidiana cruda : {er}")
    print(f"  manhattan  cruda : {mr}  | mismos índices? {er == mr}")
    print(f"  euclidiana MaxAbs: {es}")
    print(f"  manhattan  MaxAbs: {ms}  | mismos índices? {es == ms}")
    print(f"  crudo vs MaxAbs (euclidiana) mismos? {er == es}")

# Lectura basada en la salida real: ¿mismo salario? ¿qué rasgo queda?
print("\nDiagnóstico de aportes (euclidean_raw, vecinos con dist>0):")
same_salary_flags = []
remain_shares = []
for qi, q in enumerate(queries):
    for nb in nn["euclidean_raw"][qi]:
        if nb.distance <= 0:
            continue
        same_sal = abs(float(nb.features[2]) - float(q[2])) < 1e-9
        same_salary_flags.append(same_sal)
        print(
            f"  consulta {qi+1} → idx={nb.index}: mismo salary? {same_sal}; "
            f"shares={as_plain(COLUMN_NAMES, nb.shares, ndigits=4)}"
        )
        remain_shares.append(nb.shares)
        break

if same_salary_flags:
    frac_same = sum(same_salary_flags) / len(same_salary_flags)
    print(f"  fracción de primeros vecinos (dist>0) con el mismo salary: {frac_same:.0%}")
if remain_shares:
    arr = np.vstack(remain_shares)
    # Entre vecinos con salary idéntico, el aporte de salary es 0; mira el resto.
    print("  media de shares en esos vecinos:", as_plain(COLUMN_NAMES, arr.mean(axis=0), ndigits=4))
    # Excluye salary para nombrar el rasgo residual dominante cuando salary empató.
    residual = arr.copy()
    residual[:, 2] = 0.0
    row_sums = residual.sum(axis=1, keepdims=True)
    residual = np.where(row_sums > 0, residual / row_sums, residual)
    mean_res = residual.mean(axis=0)
    dom = COLUMN_NAMES[int(np.argmax(mean_res))]
    print(
        f"  tras empatar salary, rasgo residual dominante (renormalizado): "
        f"{dom} ({mean_res.max():.2%})"
    )



### Lectura T1 (Manhattan vs Euclidiana y escalado)

- **Sin escalar (salida real):** en `euclidean_raw`, los primeros vecinos con \(dist>0\) suelen compartir **exactamente el mismo `salary`** que la consulta (shares de `salary` = 0). La distancia restante la cargan edad / familiares (véase el rasgo residual dominante impreso arriba). Eso es la forma en que el salario **domina** sin escalar: primero filtra por sueldo (valores grandes / discretos), y solo después importan las demás columnas. Manhattan vs Euclidiana en crudo **no** siempre devuelven los mismos índices (p. ej. consultas 1 y 3).
- **Con MaxAbs:** los índices de vecinos cambian respecto al escenario crudo (crudo ≠ MaxAbs en las tres consultas). Al igualar escalas, deja de bastar “mismo salario” para ser vecino.
- **Conclusión operativa:** la euclidiana sin escalar no es usable para look-alikes de marketing; hay que escalar (o una métrica que no privilegie el eje del sueldo).


## T2 — Clasificación binaria (P5)

Objetivo: `insurance_benefits > 0`. Partición 70/30, semilla 42. MaxAbs ajustado solo en train.

Baselines dummy: siempre 0; probabilidad empírica; 0.5; siempre 1.


In [ ]:
rng = np.random.RandomState(SEED)
n = X.shape[0]
perm = rng.permutation(n)
split = int(0.7 * n)
train_idx, test_idx = perm[:split], perm[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_bin[train_idx], y_bin[test_idx]

print("train", X_train.shape, "test", X_test.shape)
print("tasa positiva global =", round(stats["positive_rate"], 4))
print(
    "tasa positiva train/test =",
    round(float(y_train.mean()), 4),
    round(float(y_test.mean()), 4),
)

scale = np.max(np.abs(X_train), axis=0)
scale = np.where(scale == 0.0, 1.0, scale)
X_train_s = X_train / scale
X_test_s = X_test / scale

ks = range(1, 11)
f1_raw = knn_f1_sweep(X_train, y_train, X_test, y_test, ks=ks)
f1_scaled = knn_f1_sweep(X_train_s, y_train, X_test_s, y_test, ks=ks)

print("k | F1 crudo | F1 MaxAbs")
for k, a, b in zip(ks, f1_raw, f1_scaled):
    print(f"{k:2d} | {a:.4f} | {b:.4f}")

dummy_metrics = dummies(y_test, seed=SEED)
print("\nBaselines dummy sobre etiquetas de test:")
dummy_f1s = {}
for name, m in dummy_metrics.items():
    dummy_f1s[name] = m["f1"]
    print(
        f"  {name}: F1={m['f1']:.4f}  "
        f"CM(tn,fp,fn,tp)=({m['tn']:.3f},{m['fp']:.3f},{m['fn']:.3f},{m['tp']:.3f})"
    )

best_dummy = max(dummy_f1s.values())
worst_dummy = min(dummy_f1s.values())
print(f"\nMejor F1 dummy = {best_dummy:.4f} | peor F1 dummy = {worst_dummy:.4f}")
print(f"F1 kNN crudo   min/max = {min(f1_raw):.4f} / {max(f1_raw):.4f}")
print(f"F1 kNN MaxAbs  min/max = {min(f1_scaled):.4f} / {max(f1_scaled):.4f}")
print(f"kNN MaxAbs supera al mejor dummy en todos los k? {all(v > best_dummy for v in f1_scaled)}")
print(f"kNN crudo puede quedar por debajo del mejor dummy? {any(v < best_dummy for v in f1_raw)}")
print(f"F1 MaxAbs k=1 → k=10: {f1_scaled[0]:.4f} → {f1_scaled[-1]:.4f}")

fig_f1 = f1_vs_k(f1_raw, f1_scaled)
plt.show()


### Comparación kNN vs dummies (números de la celda anterior)

- Un kNN **entrenado con MaxAbs** puede **superar** a los cuatro dummies: su F1 se mantiene alto (aprox. \(0.97\) en \(k=1\) con una **ligera baja** hacia \(\sim 0.92\) en \(k=10\); no lo llamamos “estable”).
- Un kNN **sin escalar**, sobre todo con \(k\) grandes, **puede quedar por debajo** del mejor dummy (y lejos del MaxAbs): sin escalar, el salario vuelve a dominar la vecindad y el clasificador se degrada.
- Respuesta directa: **sí** puede un modelo entrenado (kNN escalado) batir a los dummies; **sí** puede un modelo mal preparado (kNN crudo, \(k\) grande) hacerlo peor que un baseline trivial.


## T3 — Regresión lineal OLS (P6, P7)

Ajuste en el **70 % de entrenamiento**; RMSE y residuos sobre el **30 % de prueba** (semilla 42). Comparación crudo vs MaxAbs (escalas solo del train). Predicciones **sin truncar**.


In [ ]:
# Misma semilla 42; partición independiente sobre el target de conteo y.
rng_reg = np.random.RandomState(SEED)
perm_r = rng_reg.permutation(n)
split_r = int(0.7 * n)
tr_r, te_r = perm_r[:split_r], perm_r[split_r:]

X_tr, X_te = X[tr_r], X[te_r]
y_tr, y_te = y[tr_r], y[te_r]

# MaxAbs: fit solo en train
scale_r = np.max(np.abs(X_tr), axis=0)
scale_r = np.where(scale_r == 0.0, 1.0, scale_r)
X_tr_s, X_te_s = X_tr / scale_r, X_te / scale_r


def ols_train_predict(X_fit, y_fit, X_eval, y_eval):
    """Ajusta OLS en train; predice y calcula RMSE en eval (mismo espacio de rasgos)."""
    fit = linreg_ols(X_fit, y_fit, scale=False)
    A_te = _design_matrix(np.asarray(X_eval, dtype=np.float64))
    preds = A_te @ fit.weights
    rmse = float(np.sqrt(np.mean((y_eval.astype(np.float64) - preds) ** 2)))
    return fit, preds, rmse


ols_raw_fit, pred_te_raw, rmse_te_raw = ols_train_predict(X_tr, y_tr, X_te, y_te)
ols_sc_fit, pred_te_sc, rmse_te_sc = ols_train_predict(X_tr_s, y_tr, X_te_s, y_te)

# Baseline trivial: media del train predicha para todo el test
baseline_pred = np.full_like(y_te, fill_value=float(y_tr.mean()), dtype=np.float64)
rmse_baseline = float(np.sqrt(np.mean((y_te.astype(np.float64) - baseline_pred) ** 2)))

print("pesos (crudo, intercepto primero):", np.round(ols_raw_fit.weights, 6))
print(f"RMSE test crudo   = {rmse_te_raw:.6f}")
print(f"RMSE test MaxAbs  = {rmse_te_sc:.6f}")
print(f"|RMSE crudo − MaxAbs| = {abs(rmse_te_raw - rmse_te_sc):.3e}")
print(f"RMSE baseline (media train) en test = {rmse_baseline:.6f}")
print(
    "rango predicciones test (crudo): "
    f"[{float(pred_te_raw.min()):.4f}, {float(pred_te_raw.max()):.4f}]"
)
print("fracción predicciones negativas (test) =", float(np.mean(pred_te_raw < 0)))
print(
    "fracción predicciones no enteras (test) =",
    float(np.mean(np.abs(pred_te_raw - np.round(pred_te_raw)) > 1e-8)),
)

fig_rvf = residuals_vs_fitted(y_te, pred_te_raw)
plt.show()

resid_te = y_te.astype(np.float64) - pred_te_raw
fig_age = age_binned_residual_means(X_te[:, 1], resid_te)
plt.show()


### Lectura T3

- Los dos RMSE de test (crudo y MaxAbs) coinciden (salvo ruido numérico): MaxAbs es un reescalado lineal por columna; el hiperplano OLS en el espacio ampliado con intercepto produce las **mismas predicciones** \(\hat y\) (la reescala se absorbe en los pesos). Por eso el RMSE no cambia.
- El patrón por bins de edad **no** se anula solo porque la edad tenga peso distinto de cero. Los mínimos cuadrados equilibran residuales **en promedio** sobre la muestra; no capturan la relación no lineal / escalonada edad→beneficios que muestran las barras.
- El rango de predicciones en test es el impreso arriba (en esta corrida ≈ \(-0.35\) a \(1.19\); no “hasta 1.5”). Hay predicciones negativas y no enteras: imposibles como conteos.
- Frente a la **media del train** como baseline en el mismo test, el OLS mejora algo el RMSE pero el ajuste sigue siendo pobre para un conteo inflado en cero: el modelo lineal **no** es adecuado como predictor de beneficios.



## T4 — Ofuscación de datos (P8)

Procedimiento visible: dibujar \(P\), verificar \(\det P \neq 0\), transformar \(X_{\mathrm{obf}}=XP\), ajustar OLS, recuperar \(X_{\mathrm{obf}}P^{-1}\).

La columna de unos del intercepto se construye **dentro** de `linreg_ols` y **nunca** se multiplica por \(P\).


In [ ]:
# Pipeline visible (además de obfuscate, que encapsula lo mismo).
print("1) Dibujar P invertible con seed 42")
P_vis = _draw_invertible(SEED)
det_P = float(np.linalg.det(P_vis))
print("   P shape:", P_vis.shape, "| det(P) =", det_P)
assert abs(det_P) > 1e-8

print("2) Transformar bloque de rasgos: X_obf = X @ P")
X_obf_vis = X @ P_vis

print("3) Ajuste OLS original vs ofuscado")
ols_o = linreg_ols(X, y, scale=False)
ols_p = linreg_ols(X_obf_vis, y, scale=False)
r2_o = _r2(y, ols_o.predictions)
r2_p = _r2(y, ols_p.predictions)
print(f"   RMSE original   = {ols_o.rmse:.6f} | R² = {r2_o:.6f}")
print(f"   RMSE ofuscado   = {ols_p.rmse:.6f} | R² = {r2_p:.6f}")
print(f"   |ΔRMSE| = {abs(ols_o.rmse - ols_p.rmse):.3e} | max|Δpred| = {np.max(np.abs(ols_o.predictions - ols_p.predictions)):.3e}")

print("4) Recuperar: X_rec = X_obf @ inv(P)")
X_rec_vis = X_obf_vis @ np.linalg.inv(P_vis)
rec_err = float(np.max(np.abs(X - X_rec_vis)))
print(f"   max|X − X_rec| = {rec_err:.3e}")

# Tres clientes: original / ofuscado / recuperado
demo_idx = [0, 100, 250]
demo_idx = [i for i in demo_idx if i < X.shape[0]]
print("\nEjemplos de clientes (original → ofuscado → recuperado):")
for i in demo_idx:
    print(f"\n  cliente fila {i}")
    print("    original  :", as_plain(COLUMN_NAMES, X[i], ndigits=4))
    print("    ofuscado  :", as_plain([f"z{j}" for j in range(4)], X_obf_vis[i], ndigits=4))
    print("    recuperado:", as_plain(COLUMN_NAMES, X_rec_vis[i], ndigits=4))

print(
    "\n¿Se puede adivinar edad o ingreso tras la transformación? "
    "No de forma fiable: las coordenadas ofuscadas son mezclas lineales densas "
    "(valores fuera de las escalas naturales de age/salary). Sin P, no hay lectura directa."
)

# Mismo resultado vía obfuscate (API compacta)
obf = obfuscate(X, y)
assert np.allclose(obf.P, P_vis)
assert obf.pred_diff < 1e-8
assert obf.rmse_diff < 1e-8
assert np.allclose(X, obf.X_recovered, atol=1e-8)
print("\nobfuscate() confirma: pred_diff / rmse_diff < 1e-8; recuperación ≈ X (atol 1e-8).")


### Demostración analítica

Partimos de \(w=(X^\top X)^{-1}X^\top y\) y \(w_P=\bigl[(XP)^\top XP\bigr]^{-1}(XP)^\top y\) con \(P\) invertible.

1. \((XP)^\top=P^\top X^\top\), \((XP)^\top XP=P^\top X^\top X\,P\), \(\bigl[(XP)^\top XP\bigr]^{-1}=P^{-1}(X^\top X)^{-1}(P^\top)^{-1}\).
2. \(w_P=P^{-1}w\).
3. \(\hat y_P=(XP)w_P=Xw=\hat y\).
4. Mismos residuales ⇒ mismo RMSE (salvo ruido de punto flotante).

**Nota:** si la matriz de diseño incluye unos, se usa \(Q=\mathrm{diag}(1,P)\) sobre \([\mathbf{1}\mid X]\); aquí \(P\) actúa solo sobre el bloque de rasgos.


## Conclusiones (P9)

| Tarea | Hallazgo + evidencia | Advertencia |
|-------|----------------------|-------------|
| **T1** | Sin escalar, los vecinos euclidianos empatan primero en `salary` (shares de sueldo ≈ 0; residual en edad/familia); Manhattan≠Euclidiana en índices a menudo. Con MaxAbs cambian los vecinos. | Euclidiana cruda ≠ herramienta de marketing. |
| **T2** | kNN MaxAbs supera a los dummies (F1 alto, ≈0.97→≈0.92 al subir \(k\)). kNN crudo con \(k\) grande puede quedar **por debajo** del mejor dummy. | Sin selección de modelo ni claim de producción. |
| **T3** | RMSE de test crudo = MaxAbs; ambos mejoran algo la media-train, pero predicciones negativas/fraccionarias y residuales por edad muestran mal ajuste. | OLS no es predictor adecuado de este conteo. |
| **T4** | RMSE y \(R^2\) coinciden entre \(X\) y \(XP\); 3 clientes muestran original/ofuscado/recuperado. Recuperación exacta **hasta ruido de punto flotante** (`max\|X−X_rec\|` ~ 1e-10). | Quien conoce \(P\) deshace la ofuscación; no es cumplimiento normativo. |

Checklist: T1 cuatro escenarios \(k=5\) + shares + filas reales; T2 F1-vs-\(k\) + dummies; T3 RMSE hold-out + residuales test + baseline; T4 pipeline \(P\) visible + métricas + ejemplos. Sin claims de producción, regulación ni selección de modelo.

